In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPRegressor

import os

# Cria as pastas caso não existam
os.makedirs("resultados/graficos", exist_ok=True)
os.makedirs("resultados/stats", exist_ok=True)
os.makedirs("resultados/residuos", exist_ok=True)

# Arquivos de entrada


data = {
    '0.2': '/home/nathalya/Documents/Projeto_Rede_Neural/database/data_occ0_2.csv',
    '0.5': '/home/nathalya/Documents/Projeto_Rede_Neural/database/data_occ0_5.csv',
    '0.7': '/home/nathalya/Documents/Projeto_Rede_Neural/database/data_occ0_7.csv',
    '0.9': '/home/nathalya/Documents/Projeto_Rede_Neural/database/data_occ0_9.csv'
}


# Modelo base


base_model = MLPRegressor(
    solver='adam',
    max_iter=300,
    random_state=42
)


# Grid Search


param_grid = {
    'hidden_layer_sizes': [(5,),(7,),(10,),(15,),(7, 5),(10, 5)],
    'activation': ['relu','tanh'],
    'alpha': [1e-5,1e-4,1e-3],
    'learning_rate_init': [0.0001,0.001]
}


# Armazenamento


results = []
residuals_dict = {}


# Loop OCC


for occ, path in data.items():

    print(f"\n{'='*50}")
    print(f"OCC = {occ}")
    print(f"{'='*50}")

  
    # Leitura dos dados
   

    df = pd.read_csv(path)

    X = df[['sample(0)', 'sample(1)', 'sample(2)',
            'sample(3)', 'sample(4)', 'sample(5)',
            'sample(6)']].values

    y = df['AmplitudeSample(4)'].values

    # Holdout
  

    X_temp, X_test, y_temp, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

    # Grid Search
   

    grid = GridSearchCV(
        estimator=base_model,
        param_grid=param_grid,
        cv=10,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_temp, y_temp)

  
    # Melhor modelo
    

    final_model = grid.best_estimator_

    final_model.fit(X_temp, y_temp)

  
    # Teste final
  

    y_test_pred = final_model.predict(X_test)

    test_residuals = y_test - y_test_pred

    test_mae = np.mean(np.abs(test_residuals))
    test_std = np.std(test_residuals)

    residuals_dict[occ] = test_residuals

    # Curva de aprendizado
  

    plt.figure(figsize=(8, 5))

    plt.plot(final_model.loss_curve_)

    plt.xlabel('Épocas')
    plt.ylabel('Loss')
    plt.title(f'Curva de Aprendizado - OCC {occ}')
    plt.grid(True)

    plt.savefig(
        f'curva_aprendizado_OCC_{occ}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close()

  
    # Real x Predito
  

    a, b = np.polyfit(y_test, y_test_pred, 1)

    x_fit = np.linspace(
        y_test.min(),
        y_test.max(),
        100
    )

    y_fit = a * x_fit + b

    plt.figure(figsize=(8, 6))

    plt.scatter(
        y_test,
        y_test_pred,
        alpha=0.2,
        s=15,
        label='Predições'
    )

    plt.plot(
        x_fit,
        x_fit,
        '--',
        color='black',
        linewidth=3,
        label='Ideal'
    )

    plt.plot(
        x_fit,
        y_fit,
        '-',
        color='red',
        linewidth=3,
        label=f'Fit: y={a:.3f}x + {b:.3f}'
    )

    plt.xlabel('Amplitude Real')
    plt.ylabel('Amplitude Predita')
    plt.title(f'Real vs Predito - OCC {occ}')

    plt.legend()
    plt.grid(True)

    plt.savefig(
        f'real_vs_predito_OCC_{occ}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()
    plt.close()
  
    # Resultados
  

    results.append({
        'Occupancy': float(occ),

        'Best_CV_MAE': -grid.best_score_,

        'Test_MAE': test_mae,
        'Test_STD': test_std,

        'Best_hidden_layers':
            str(grid.best_params_['hidden_layer_sizes']),

        'Best_activation':
            grid.best_params_['activation'],

        'Best_alpha':
            grid.best_params_['alpha'],

        'Best_learning_rate':
            grid.best_params_['learning_rate_init'],

        'Best_n_iter':
            final_model.n_iter_,

        'Final_Loss':
            final_model.loss_
    })

  

    print("\nMelhores parâmetros:")

    for k, v in grid.best_params_.items():
        print(f"{k}: {v}")

    print(f"\nCV MAE: {-grid.best_score_:.6f}")
    print(f"Test MAE: {test_mae:.6f}")
    print(f"Test STD: {test_std:.6f}")


# Salvar métricas


stats_df = pd.DataFrame(results)

stats_df.to_csv(
    "rn_stats.csv",
    index=False
)


# Salvar resíduos

rn_residuals = pd.DataFrame(
    residuals_dict
)

rn_residuals.to_csv(
    "rn_residuals.csv",
    index=False
)

